# Implementing a Merkle Tree

To implement a Merkle tree on a set of files, we need to define:

1) A function for hashing the files.
2) A function for generating a membership proof given a file hash.
3) A function for verifying the membership proofs.

Let's start by importing the library.

In [1]:
from Crypto.Hash import SHA256

We first define the function for hashing each file. We read the file in chunks of 8 KiB.

In [2]:
def hash_file(file_path: str) -> bytes:
    chunk_size = 8192 # We read the file in chunks of 8 KiB
    h = SHA256.new()
    with open(file_path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.digest()

Let's test the function. 
On UNIX-based system, the command

```shasum -a 256 data/othello_TXT_FolgerShakespeare.txt```

should produce the same output.

In [3]:
hash_file("data/othello_TXT_FolgerShakespeare.txt").hex()

'6b654ec71126492b0061fd8c995f6a5f511360841fadac07e7cd6810f766d722'

## Building the tree

Let us define a function for creeating a tree from scratch.

The function takes a list of file paths as input and returns the full tree.

The full tree is represented as a list of lists. Each list represents a level of the tree.

In [4]:
def build_merkle_tree(file_list: list[str]) -> list[list[bytes]]:
    # First, we compute the leaf hashes
    leaf_hashes = []
    for file_path in file_list:
        leaf_hashes.append(hash_file(file_path))
    # Then we start building the internal nodes level by level, iteratively.
    tree = [leaf_hashes]
    current_level = leaf_hashes
    while len(current_level) > 1:
        next_level = []
        for i in range(0, len(current_level), 2):
            left = current_level[i]
            # When the level has an odd number of nodes, we duplicate the last one
            # in order to make it even.
            right = current_level[i + 1] if i + 1 < len(current_level) else left
            h = SHA256.new()
            h.update(left + right)
            parent = h.digest()
            next_level.append(parent)
        tree.append(next_level)
        current_level = next_level
    return tree

Let's use our function to build the tree starting from the text files in the `data` directory.

The directory contains 5 distinct text files.

In [5]:
file_list = [
    "data/hamlet_TXT_FolgerShakespeare.txt",
    "data/macbeth_TXT_FolgerShakespeare.txt",
    "data/othello_TXT_FolgerShakespeare.txt",
    "data/romeo-and-juliet_TXT_FolgerShakespeare.txt",
    "data/the-merchant-of-venice_TXT_FolgerShakespeare.txt"
]

# Build the tree.
tree = build_merkle_tree(file_list)

# Print the tree from the root to the leaves, level by level.
for level in reversed(tree):
    print([h.hex() for h in level])


['9740f5429bd8cb9de091a85cb0155f58e5df78946f97e4094661667abca65b06']
['6001c6a5fd82f3981475c5ed6edf0cd84280f475c338c404ebffac6fada9dc39', 'd10c90ba55bd2c8861ed8e89fe6f19cfb071a9eac1151b36518eb9fdf1c25b1a']
['b5c61d481499963a8f8eaca1199c8c7264b72634996a9089048f7747f4218073', 'ce69e9f8f26faeedb004a47b715da01133da582bd215aff60ce848a51148b7a3', '1930b841beaab0d52577af7362e4ed3e6e2bdd6886f591f922594ba47024e6c6']
['3d9b03e4051a202ae263f65cd4d24371af5ce8655629a73bddf87aad253db5f3', '2189f6b93022baf7214f6f02c9b6798129f0fb94accc634df26af05fe9b887b8', '6b654ec71126492b0061fd8c995f6a5f511360841fadac07e7cd6810f766d722', '37b24f14cff006b68b0b7783d00f48c1a7ecb531865a8e01c42da6953206398c', '48daa206aca51dfc35bf5bc204715d22099c61d8e3c0e7a24f3602c638ea90d0']


The following utility functions return:

- the hash of the root of the tree;
- the list of leaves.

In [6]:
def get_merkle_root(tree: list[list[bytes]]) -> bytes:
    return tree[-1][0]

def get_leaves(tree: list[list[bytes]]) -> list[bytes]:
    return tree[0]

In [9]:
print("Root:", get_merkle_root(tree).hex())
print("Leaves:", [l.hex() for l in get_leaves(tree)])

Root: 9740f5429bd8cb9de091a85cb0155f58e5df78946f97e4094661667abca65b06
Leaves: ['3d9b03e4051a202ae263f65cd4d24371af5ce8655629a73bddf87aad253db5f3', '2189f6b93022baf7214f6f02c9b6798129f0fb94accc634df26af05fe9b887b8', '6b654ec71126492b0061fd8c995f6a5f511360841fadac07e7cd6810f766d722', '37b24f14cff006b68b0b7783d00f48c1a7ecb531865a8e01c42da6953206398c', '48daa206aca51dfc35bf5bc204715d22099c61d8e3c0e7a24f3602c638ea90d0']


## Generating a membership proof

Let's see how we can generate a membership proof given a Merkle tree and a target leaf hash.

In [11]:
def get_proof(tree: list[list[bytes]], target_hash: bytes) -> list[tuple[bytes, str]]:
    
    # First, we locate the hash among the leaves (this step can be optimized).
    leaf_hashes = tree[0]
    try:
        index = leaf_hashes.index(target_hash)
    except ValueError:
        raise ValueError("Error: target hash not found among tree leaves!")
        
    # To construct the proof, we iterate through the tree levels.
    proof = []
    for level in tree[:-1]:
        # Determine whether the target node is a left or right child w.r.t. its parent.
        is_right_node = index % 2 
        # Then, according to this, we determine the position of its sibling.
        sibling_index = index - 1 if is_right_node else index + 1
        
        # This handles the case where the current node has no sibling.
        # In that case, we take as sibling hash the hash of the current node.
        # In other words, we duplicate the node (as done during tree construction).
        if sibling_index >= len(level):
            sibling_hash = level[index]  # duplicate case
        else:
            sibling_hash = level[sibling_index]
            
        # In the proof, we store the sibling hash and where it is located
        # w.r.t. our node (i.e., to the left or to the right).
        direction = 'left' if is_right_node else 'right'
        proof.append((sibling_hash, direction))
        
        # We move up one level and map the node to its parent index.
        index = index // 2
    return proof

Let's construct a proof for the Othello play. First, we need to compute the file digest. Then, we can generate a proof.

In [17]:
target_file_path = "data/othello_TXT_FolgerShakespeare.txt"

# Compute the digest while reading file chunks.
chunk_size = 8192
h = SHA256.new()
with open(target_file_path, "rb") as f:
    while chunk := f.read(chunk_size):
        h.update(chunk)
target_leaf = h.digest()
print("File digest:", h.hexdigest())

# Compute the proof.
proof = get_proof(tree, target_leaf)
proof

File digest: 6b654ec71126492b0061fd8c995f6a5f511360841fadac07e7cd6810f766d722


[(b'7\xb2O\x14\xcf\xf0\x06\xb6\x8b\x0bw\x83\xd0\x0fH\xc1\xa7\xec\xb51\x86Z\x8e\x01\xc4-\xa6\x952\x069\x8c',
  'right'),
 (b'\xb5\xc6\x1dH\x14\x99\x96:\x8f\x8e\xac\xa1\x19\x9c\x8crd\xb7&4\x99j\x90\x89\x04\x8fwG\xf4!\x80s',
  'left'),
 (b'\xd1\x0c\x90\xbaU\xbd,\x88a\xed\x8e\x89\xfeo\x19\xcf\xb0q\xa9\xea\xc1\x15\x1b6Q\x8e\xb9\xfd\xf1\xc2[\x1a',
  'right')]

## Verifying a membership proof

The verification function reconstructs the tree by traversing the proof array. 
Once the root is obtained, it is compared with the original one.

In [23]:
def verify_proof(leaf_hash: bytes, proof: list[tuple[bytes, str]], original_root: bytes) -> bool:
    computed_hash = leaf_hash
    for sibling_hash, direction in proof:
        # The 'direction' field tells us how to rebuild the parent node hash from the current sibling.
        # If direction is 'left' then we should hash the "sibling || node" string.
        if direction == 'left':
            computed_hash = SHA256.new(sibling_hash + computed_hash).digest()
        # If direction is 'right' then we should hash the "node || sibling" string.
        else:
            computed_hash = SHA256.new(computed_hash + sibling_hash).digest()
    return computed_hash == original_root

To verify the previous proof, we need also the target leaf hash (which we can already compute) and the global tree root hash (assumed to be public).

In [24]:
verify_proof(target_leaf, proof, get_merkle_root(tree))

True